# A Simple RAG from Scratch

**RAG = Retrieval-Augmented Generation.**
An LLM only knows what was in its training data. If you ask it about *your* documents
(a course handbook, internal wiki, last week's meeting notes), it will either say
"I don't know" or – worse – make something up.

RAG fixes this with a very simple trick:

```
                 ┌──────────────┐
  question ────► │  RETRIEVER   │ ──► the k most relevant text chunks
                 └──────────────┘                 │
                                                  ▼
                 ┌──────────────────────────────────────────┐
                 │ PROMPT = "Answer using ONLY this context: │
                 │          <chunks>  Question: <question>"  │
                 └──────────────────────────────────────────┘
                                                  │
                                                  ▼
                 ┌──────────────┐
                 │     LLM      │ ──► answer (grounded in your documents)
                 └──────────────┘
```

In this notebook we build every piece ourselves, in about 100 lines of Python:

1. **Corpus** – a handful of documents
2. **Chunking** – split them into small pieces
3. **Embedding** – turn each chunk into a vector of numbers
4. **Retrieval** – find the chunks whose vectors are closest to the question's vector
5. **Generation** – hand those chunks to an LLM and let it answer

Along the way we compare the LLM's answer *with* and *without* retrieval.

## 0. Setup

The notebook has two "switches" so it works in any classroom situation:

| Switch | Options | Notes |
|---|---|---|
| `EMBEDDING_BACKEND` | `"tfidf"` (no download, no GPU) or `"sentence-transformers"` (real multilingual neural embeddings, ~470 MB model download on first run) | Both are drop-in; the rest of the notebook does not change. |
| `LLM_BACKEND` | `"none"` (just prints the prompt – no API key needed), `"anthropic"`, `"openai"`, `"ollama"` (local) | Start with `"none"` to understand the pipeline, then switch to a real model. |

Uncomment the `pip install` lines you need.

In [ ]:
# Core dependencies (usually already present in Anaconda / Colab):
# %pip install numpy scikit-learn

# Optional – pick what you want to use:
# %pip install sentence-transformers      # neural embeddings
# %pip install anthropic                  # Claude API
# %pip install openai                     # OpenAI API
# %pip install ollama                     # local models via https://ollama.com

EMBEDDING_BACKEND = "tfidf"      # "tfidf"  or  "sentence-transformers"
LLM_BACKEND       = "none"       # "none", "anthropic", "openai", "ollama"

# API keys (only needed for anthropic / openai). Better: set them as environment variables.
import os
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["OPENAI_API_KEY"]    = "sk-..."

import re, textwrap
import numpy as np

## 1. The corpus

A small knowledge base built from the public website **[aikurs.ch](https://aikurs.ch)** – the
"Machine Learning Masterclass" (overview, agenda, FAQ, cancellation policy, team). The texts are
in German. These are niche facts (dates, prices, policies of one small course) that an LLM will not
know reliably from training – so a correct answer *must* come from retrieval.

> 💡 Later, replace this with your own documents (course notes, PDFs converted to text, …).

In [ ]:
DOCUMENTS = {
"ueberblick.md": '''
Die Machine Learning Masterclass von AIKurs.ch ist ein zweitägiger Intensiv-Workshop zu den
technischen Grundlagen von Machine Learning und Sprachmodellen (LLMs). Zielgruppe sind Senior
Engineers, Tech Leads, CTOs und Tech-Founder. Ziel des Kurses ist es, ML-Projekte beurteilen und
fundierte Entscheidungen treffen zu können, nicht ML-Systeme von Grund auf selbst zu bauen.
Das Format besteht zu 50 Prozent aus Theorie im Klassenzimmer und zu 50 Prozent aus Hands-on-Coding.
Der Workshop findet in Wallisellen bei Zürich statt. Die Gruppen sind klein: 10 bis 12 Teilnehmende
werden von zwei technischen Coaches betreut. Der Preis beträgt CHF 1780 exklusive Mehrwertsteuer.
Die öffentlichen Durchführungen 2026 finden am 7. und 8. Mai, am 2. und 3. Juli sowie am
11. und 12. November statt. Am Freitag, 6. November 2026 um 15:30 Uhr gibt es eine kostenlose
Online-Infoveranstaltung. AIKurs.ch ist eine Initiative der Firmen VeeMG GmbH, adnexo GmbH und Sparkner AG.
''',
"agenda.md": '''
Tag 1 der Masterclass beginnt mit der Geschichte und der Richtung des aktuellen KI-Booms. Danach
folgen die Grundlagen von Machine Learning und Large Language Models. In einem Lab werden klassische
ML-Algorithmen praktisch ausprobiert. Am Nachmittag geht es um MLOps, Deployment und die damit
verbundenen Risiken sowie um die Frage, was erfolgreiche ML-Projekte von gescheiterten unterscheidet.
Tag 2 behandelt neuronale Netze und Deep Learning sowie die Funktionsweise von LLMs. Ein weiterer
Block schlägt die Brücke zwischen ML und Business. Es gibt zwei Labs: ein Beispiel zu einer
Kreditentscheidung (Credit Decision) und ein LLM-Lab. Am zweiten Tag lernen die Teilnehmenden
ausserdem, wie sie ML-Projekte intern durchbringen und eigene Use Cases skizzieren.
Was die Teilnehmenden mitnehmen: ein praktisches Verständnis, wie Modelle trainiert werden und was
unter der Haube passiert, die Fähigkeit zu beurteilen, welche ML- und LLM-Use-Cases machbar sind
und wo die Fallstricke liegen, sowie eine konkrete To-do-Liste für die Integration von Modellen
oder LLM-Agenten in die eigene Software.
''',
"faq-teilnahme.md": '''
Ist dieser Kurs das Richtige für mich? Der Kurs passt, wenn du ein Team führst, das ML einsetzt,
über Budget für ML-Projekte entscheidest oder ML verstehen willst, ohne täglich selbst zu
programmieren. Er passt nicht, wenn du ML-Engineer werden willst; dafür wird der CAS ML4SE an der
OST empfohlen. Ohne jede technische Vorbildung ist der Kurs ebenfalls nicht geeignet.
Welche Vorkenntnisse brauche ich? Du solltest Erfahrung in der Softwareentwicklung haben und selbst
programmieren können. In den Lab-Sessions arbeitet die Gruppe gemeinsam an Code, den du verstehen
sollst. Grundlegende Programmierkenntnisse genügen, du musst kein Profi sein: Die Coding-Sessions
sind angeleitet und sollen das Verständnis fördern, nicht Perfektion verlangen.
Welches Equipment brauche ich? Du benötigst einen Laptop, idealerweise mit einer Docker-Umgebung.
Bei eingeschränkten Firmenlaptops laufen die Labs im Notfall auch auf Google Colab.
Erhalte ich Unterlagen? Ja, du erhältst ein Repository und nimmst alle Materialien mit nach Hause,
um selbstständig weiterzulernen. Es gibt auch eine Teilnahmebestätigung.
''',
"faq-organisation.md": '''
Warum dauert der Workshop zwei Tage statt einen? Machine Learning ist zu komplex, um es an einem Tag
wirklich zu verstehen. Zwei Tage ermöglichen ausführliche Hands-on-Sessions und einen tieferen
Einstieg in MLOps. Kann ich nur einen Tag teilnehmen? Nein, die öffentliche Masterclass ist als
Zwei-Tages-Format konzipiert, weil die Inhalte aufeinander aufbauen. Für firmeninterne Trainings
gibt es zusätzlich ein Ein-Tages-Kompaktformat.
Kann der Workshop firmenintern durchgeführt werden? Ja, es gibt ein In-House-Angebot für Teams von
6 bis 12 Personen, im eigenen Büro oder remote, mit optionalem Follow-up-Coaching. Dabei können die
spezifischen Use Cases des Unternehmens berücksichtigt werden; für ein Angebot muss man den Anbieter
kontaktieren. Wie viele Personen können maximal teilnehmen? Maximal 12 Teilnehmende, damit intensive
Betreuung und echte Diskussionen statt Frontalunterricht möglich sind.
Was unterscheidet diese Masterclass von anderen ML-Kursen? Die Dozierenden sind Praktiker, die ML
bereits in echte Produkte gebracht haben. Der Kurs verbindet drei Perspektiven: Theorie, Umsetzung
und Business Case. Man lernt nicht nur, wie ML funktioniert, sondern auch, welche Projekte sich lohnen.
''',
"stornierung.md": '''
Stornierungsbedingungen der Machine Learning Masterclass: Bis vier Wochen vor dem Termin ist eine
Stornierung oder Umbuchung kostenlos möglich. Danach wird eine Stornogebühr von 50 Prozent fällig.
Bei einer Absage weniger als eine Woche vor dem Termin ist der volle Preis zu bezahlen.
Eine Ersatzperson kann jederzeit anstelle der angemeldeten Person geschickt werden.
''',
"team.md": '''
Die Gründer und Dozenten Martin und Christian kennen sich seit fast acht Jahren. Martin hat den
CAS ML4SE an der OST aufgebaut; Christian wurde ihm als kompetenter Dozent aus anderen Programmen
empfohlen. Martin konzipierte einen Workshop für die Konferenz MLCon, Christian stieg als
gleichberechtigter Partner ein. Der erste gemeinsame Workshop fand 2023 in Berlin als eintägige
Veranstaltung statt. Danach traten die beiden auf MLCon-Bühnen in London, München und Berlin auf;
weitere Stationen waren die CHOpen Workshop Tage 2024 und 2025. Das Feedback der Teilnehmenden hat
das Programm stark geprägt. Philip Wels ergänzt das Team als Experte für Business Development und
Go-to-Market-Strategien; er hilft den Teilnehmenden, ML-Projekte intern zu verkaufen und aus Proofs
of Concept echte Produkte zu machen. Das Selbstverständnis des Teams: Praktiker für Praktiker, ohne
Hype-Frameworks und Marketing-Buzzwords. Teilnehmende kamen unter anderem von Swiss Life und
Schmid Elektronik.
''',
}

for name, text in DOCUMENTS.items():
    print(f"{name:20s} {len(text.split()):4d} words")

## 2. Chunking

We do not embed whole documents. We split them into **chunks** of a few sentences, because

* a retrieved chunk should be *focused* – one topic, not an entire handbook page, and
* the LLM prompt has a limited size; we want to fit several relevant chunks in.

Our chunker is deliberately simple: split into sentences, then group consecutive sentences
until a chunk reaches roughly `chunk_size` words. Neighbouring chunks **overlap** by one
sentence so that a fact that straddles a boundary is not lost.

> 🧪 Try different values of `chunk_size` later and see how retrieval quality changes.

In [ ]:
def split_sentences(text):
    text = " ".join(text.split())                       # normalise whitespace
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

def chunk_document(text, chunk_size=60, overlap_sentences=1):
    sentences = split_sentences(text)
    chunks, current = [], []
    for sentence in sentences:
        current.append(sentence)
        if sum(len(s.split()) for s in current) >= chunk_size:
            chunks.append(" ".join(current))
            current = current[-overlap_sentences:] if overlap_sentences else []
    if current and " ".join(current) not in chunks:
        chunks.append(" ".join(current))
    return chunks

# Build the list of chunks, remembering which document each one came from.
chunks = []          # list of dicts: {"text": ..., "source": ...}
for name, text in DOCUMENTS.items():
    for c in chunk_document(text, chunk_size=60):
        chunks.append({"text": c, "source": name})

print(f"{len(DOCUMENTS)} documents  ->  {len(chunks)} chunks\n")
for i, c in enumerate(chunks[:4]):
    print(f"[{i}] ({c['source']})")
    print(textwrap.fill(c["text"], 100), "\n")

## 3. Embeddings: text → vector

To find "relevant" chunks we need a way to compare a question to a chunk numerically.
An **embedding** maps a piece of text to a vector (a list of numbers) such that texts with
similar *meaning* get vectors that point in similar directions.

We offer two embedding functions with the same interface `embed(list_of_texts) -> matrix`:

* **TF-IDF** (`scikit-learn`): a classic *bag-of-words* approach. Each dimension is a word;
  the value is high if the word is frequent in this text but rare across the corpus.
  Works offline, no neural network, but only matches on *word overlap* – it does not know
  that "swimming" and "pool" are related.
* **Sentence-Transformers** (`paraphrase-multilingual-MiniLM-L12-v2`): a small neural network that
  produces 384-dimensional vectors capturing *semantics*, and it understands German. This is what
  real RAG systems use.

The vectors are **L2-normalised** (length 1), so the dot product of two vectors equals their
**cosine similarity**: 1 = identical direction, 0 = unrelated.

In [ ]:
if EMBEDDING_BACKEND == "tfidf":
    from sklearn.feature_extraction.text import TfidfVectorizer
    _vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)   # no stop-word list: corpus is German
    _vectorizer.fit([c["text"] for c in chunks])      # vocabulary comes from the corpus

    def embed(texts):
        return _vectorizer.transform(texts).toarray()  # rows are already L2-normalised

elif EMBEDDING_BACKEND == "sentence-transformers":
    from sentence_transformers import SentenceTransformer
    _model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

    def embed(texts):
        return _model.encode(texts, normalize_embeddings=True)

else:
    raise ValueError(f"Unknown EMBEDDING_BACKEND {EMBEDDING_BACKEND!r}")

# A quick look at what an embedding is:
example = embed(["Der Workshop dauert zwei Tage."])
print("vector shape:", example.shape)
nonzero = np.flatnonzero(example[0])
print(f"{len(nonzero)} of {example.shape[1]} entries are non-zero (TF-IDF vectors are sparse; neural ones are dense)")
print("first few non-zero values:", np.round(example[0][nonzero[:8]], 3))
print("vector length (should be 1.0):", round(float(np.linalg.norm(example[0])), 3))

Let's check that similarity means what we hope it means. Compare three sentences with each other:

In [ ]:
sentences = [
    "Was kostet der Kurs?",
    "Wie hoch ist der Preis der Masterclass?",
    "Welchen Laptop muss ich mitbringen?",
]
V = embed(sentences)
similarity = V @ V.T          # cosine similarity matrix, because vectors are normalised

print(" " * 44 + "".join(f"{j:>8d}" for j in range(len(sentences))))
for i, s in enumerate(sentences):
    print(f"{i}: {s:40s}" + "".join(f"{similarity[i, j]:8.2f}" for j in range(len(sentences))))

Sentences 0 and 1 ask the same thing with different words. With TF-IDF their similarity is
**zero** – they share no word at all (*kostet* vs. *Preis*). With sentence-transformers it should be
clearly positive, because the model understands that "Was kostet" ≈ "Wie hoch ist der Preis".

## 4. The vector store and retrieval

A "vector database" sounds impressive, but at this scale it is just **a matrix**:
one row per chunk. Retrieval is:

1. embed the question → vector `q`
2. compute the cosine similarity of `q` with every row (one matrix–vector product)
3. return the `k` rows with the highest score

Production systems (FAISS, Chroma, pgvector, …) do exactly this, just faster for millions of rows.

In [ ]:
# Index all chunks once.
chunk_matrix = embed([c["text"] for c in chunks])
print("index shape (chunks x dimensions):", chunk_matrix.shape)

def retrieve(question, k=3):
    q = embed([question])[0]
    scores = chunk_matrix @ q                       # cosine similarity to every chunk
    top = np.argsort(scores)[::-1][:k]              # indices of the k best scores
    return [(float(scores[i]), chunks[i]) for i in top]

def show_retrieval(question, k=3):
    print("QUESTION:", question, "\n")
    for score, c in retrieve(question, k):
        print(f"  score={score:.3f}  source={c['source']}")
        print("  " + textwrap.fill(c["text"], 96, subsequent_indent="  ")[:400], "\n")

show_retrieval("Wann findet die Masterclass statt?")

In [ ]:
show_retrieval("Was passiert, wenn ich krank werde und absagen muss?")

Notice the second question: the chunk that answers it (`stornierung.md`) talks about *Stornierung*,
*Absage* and *Ersatzperson* – it never contains *krank* or *absagen muss*. TF-IDF only sees word
overlap, so it may rank the wrong chunk first, while a neural embedding model knows that being ill
and having to cancel is about cancellation. **This is the main argument for semantic embeddings.**

## 5. Generation: the "G" in RAG

Now we assemble the prompt. The key ideas are:

* put the retrieved chunks into the prompt as **context**,
* instruct the model to answer **only from that context** and to say so when the answer is not there,
* ask it to **cite** the source file – this makes hallucinations easy to spot.

The `generate()` function below talks to whichever LLM you selected in `LLM_BACKEND`.
With `"none"` it simply returns the prompt, so you can see exactly what the model would receive.

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful assistant answering questions about the Machine Learning Masterclass of AIKurs.ch. "
    "Answer the question using ONLY the information in the provided context. "
    "Answer in the language of the question. "
    "If the context does not contain the answer, say that the website does not provide this information. "
    "Keep the answer short and mention the source file in brackets, e.g. [agenda.md]."
)

def build_prompt(question, retrieved):
    context = "\n\n".join(f"[{c['source']}]\n{c['text']}" for _, c in retrieved)
    return f"CONTEXT:\n{context}\n\nQUESTION: {question}\n\nANSWER:"

def generate(prompt, system=SYSTEM_PROMPT):
    if LLM_BACKEND == "none":
        return "(no LLM configured – this is the prompt that would be sent)\n\n" + \
               "SYSTEM: " + system + "\n\n" + prompt

    if LLM_BACKEND == "anthropic":
        import anthropic
        client = anthropic.Anthropic()                     # reads ANTHROPIC_API_KEY
        resp = client.messages.create(
            model="claude-sonnet-4-5", max_tokens=300, system=system,
            messages=[{"role": "user", "content": prompt}])
        return resp.content[0].text

    if LLM_BACKEND == "openai":
        from openai import OpenAI
        client = OpenAI()                                  # reads OPENAI_API_KEY
        resp = client.chat.completions.create(
            model="gpt-4o-mini", max_tokens=300,
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": prompt}])
        return resp.choices[0].message.content

    if LLM_BACKEND == "ollama":
        import ollama                                      # e.g. `ollama pull llama3.2`
        resp = ollama.chat(model="llama3.2",
                           messages=[{"role": "system", "content": system},
                                     {"role": "user", "content": prompt}])
        return resp["message"]["content"]

    raise ValueError(f"Unknown LLM_BACKEND {LLM_BACKEND!r}")

### Putting it all together: `rag_answer()`

This single function *is* the RAG system. Everything above was preparation.

In [ ]:
def rag_answer(question, k=3, verbose=True):
    retrieved = retrieve(question, k)
    prompt = build_prompt(question, retrieved)
    answer = generate(prompt)
    if verbose:
        print("QUESTION:", question)
        print("RETRIEVED:", ", ".join(f"{c['source']} ({s:.2f})" for s, c in retrieved))
        print("-" * 80)
        print(answer)
        print("=" * 80, "\n")
    return answer

rag_answer("Wie viele Teilnehmende hat ein Workshop maximal?")

In [ ]:
rag_answer("Was lerne ich am zweiten Tag?")

## 6. With vs. without retrieval

The whole point of RAG is visible only when a real LLM is connected
(set `LLM_BACKEND` to `"anthropic"`, `"openai"` or `"ollama"` and re-run the setup cell).

* **Without** context the model knows nothing reliable about a small Swiss course. A good model
  admits that; a weaker one invents plausible-sounding prices and dates.
* **With** context it answers correctly and cites the file.
* For a question the website does **not** cover, a well-behaved RAG system says so instead of
  guessing – because we told it to in the system prompt.

In [ ]:
question = "Was kostet die Machine Learning Masterclass von AIKurs.ch?"

print("=== WITHOUT retrieval ===")
print(generate(f"QUESTION: {question}\n\nANSWER:",
               system="You are a helpful assistant. Answer briefly."))

print("\n=== WITH retrieval (RAG) ===")
rag_answer(question)

print("=== Question the website cannot answer ===")
rag_answer("Gibt es am Kurs ein vegetarisches Mittagessen?")

## 7. Exercises

1. **Chunk size.** Set `chunk_size=20` and `chunk_size=200` in section 2 and re-run. How do the
   retrieved chunks change for *"Kann ich eine Ersatzperson schicken?"*? What is the trade-off?
2. **k.** Call `rag_answer(..., k=1)` and `k=5`. When does a larger `k` help, when does it hurt?
3. **Embeddings.** Switch `EMBEDDING_BACKEND` to `"sentence-transformers"` and re-run from section 3.
   Compare the similarity matrix in section 3 and the retrieval for *"Was passiert, wenn ich krank werde?"*.
4. **Failure analysis.** Write three questions whose answer *is* in the corpus but which the
   retriever gets wrong. Why does it fail? (Hint: synonyms, numbers, questions spanning two files.)
5. **Hallucination check.** With a real LLM, remove the sentence *"If the context does not contain
   the answer, say …"* from `SYSTEM_PROMPT` and ask about the vegetarian lunch again. What happens?
6. **Your own data.** Replace `DOCUMENTS` with your own material – `.md`/`.txt` files via
   `pathlib.Path("docs").glob("*.md")`, or PDFs via `pypdf.PdfReader(path).pages[i].extract_text()`.
7. **Language.** Ask a question in English (*"When does the course take place?"*). Which embedding
   backend still finds the German chunk, and why?

## What we skipped (and what real systems add)

| Topic | Why it matters |
|---|---|
| Better chunking (by heading, by token count) | Chunks that respect document structure retrieve better. |
| Hybrid search (keyword + vector) | Catches exact terms (product codes, names) that embeddings blur. |
| Re-ranking | A second, more expensive model re-orders the top-20 candidates before generation. |
| Metadata filtering | "Only search documents from 2025" / "only this course". |
| Evaluation | Measure retrieval recall and answer faithfulness on a test set instead of eyeballing. |
| Vector databases | FAISS, Chroma, Qdrant, pgvector – same idea as our matrix, built for millions of chunks. |